# Day 11 — Threads, Processes & the GIL, Visualized

CPython's Global Interpreter Lock lets only one thread run Python bytecode at a time. So
for CPU-bound work, threads give **no speedup** — but processes do. Let's measure it.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

def heavy(n):
    total = 0
    for i in range(n): total += i * i
    return total

TASKS = [5_000_000] * 4

def timed(runner, label):
    t = time.perf_counter(); runner(); dt = time.perf_counter() - t
    print(f'{label:22} {dt:.2f}s'); return dt

serial = timed(lambda: [heavy(n) for n in TASKS], 'serial')
threads = timed(lambda: list(ThreadPoolExecutor(4).map(heavy, TASKS)), 'threads (GIL-bound)')

## Processes sidestep the GIL

(Runs best on Linux/macOS. Each process has its own interpreter, so they run in true parallel.)

In [ ]:
import matplotlib.pyplot as plt

try:
    procs = timed(lambda: list(ProcessPoolExecutor(4).map(heavy, TASKS)), 'processes')
except Exception as e:
    print('process pool unavailable here:', e); procs = serial

labels = ['serial', 'threads', 'processes']
times = [serial, threads, procs]
plt.figure(figsize=(6, 3.5))
plt.bar(labels, times, color=['#888', '#DD8452', '#55A868'])
plt.ylabel('seconds (lower is better)')
plt.title('CPU-bound: threads ~ serial (GIL), processes win')
for i, t in enumerate(times): plt.text(i, t, f'{t:.2f}s', ha='center', va='bottom')
plt.show()

## Takeaways

- Threads don't speed up CPU-bound Python (GIL) — but they're great for **I/O-bound** work.
- Processes achieve real parallelism at the cost of not sharing memory (must pass messages).
- Rule of thumb: I/O-bound → threads/async; CPU-bound → processes.

**Now build** chunk / threaded_sum / parallel_map in `homework.py`, then `pytest -q`.